# Natural Gas Price Prediction using FinBERT

This notebook creates a pipeline using FinBERT from Hugging Face to estimate the daily change in the Henry Hub Natural Gas Spot Price based on news sentiment.

**Data Sources:**
- News data: `gs://codeml/gas_headlines.json`
- Price data: `gs://codeml/DHHNGSP.csv`

**Output:**
- CSV file: `price_predictions.csv` with columns: `today_news_date`, `today_price`, `predicted_tomorrow_price`, `actual_tomorrow_price`

## 1. Install Dependencies

In [ ]:
!pip install transformers torch protobuf==3.20.3 tqdm pandas scikit-learn

## 2. Load FinBERT Model and Define Helper Functions

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

# Load the FinBERT model and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)
print("FinBERT model and tokenizer loaded successfully.")


def get_sentiment(news_articles):
    """
    Analyzes the sentiment of a list of news articles using the FinBERT model.

    Args:
        news_articles (list of str): A list of news headlines and summaries.

    Returns:
        torch.Tensor: A tensor containing the sentiment probabilities (positive, negative, neutral) for each article.
    """
    inputs = tokenizer(news_articles, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    return probs


def aggregate_sentiment_score(sentiment_probabilities):
    """
    Aggregates sentiment probabilities into a single score.

    Args:
        sentiment_probabilities (torch.Tensor): A tensor where each row contains
                                                  the probabilities for [positive, negative, neutral].

    Returns:
        float: The aggregated sentiment score, calculated as the average of (positive - negative) scores.
    """
    scores = sentiment_probabilities[:, 1] - sentiment_probabilities[:, 2]
    aggregated_score = scores.mean()
    return aggregated_score.item()


def predict_price_change(aggregated_score, previous_price):
    """
    Estimates the price change based on the aggregated sentiment score.

    Args:
        aggregated_score (float): The aggregated sentiment score for the day.
        previous_price (float): The previous day's spot price.

    Returns:
        float: The predicted price difference.
    """
    scaling_factor = 0.15
    predicted_difference = aggregated_score * abs(aggregated_score) * scaling_factor
    return predicted_difference

## 3. Test Pipeline with Sample Data

In [ ]:
def natural_gas_pipeline(news_articles, previous_price):
    """
    Runs the full pipeline to predict natural gas price change based on news sentiment.

    Args:
        news_articles (list of str): A list of news headlines and summaries.
        previous_price (float): The previous day's spot price.

    Returns:
        float: The predicted price difference.
    """
    sentiment_probabilities = get_sentiment(news_articles)
    print(f"Step 1: Sentiment Probabilities Calculated:\n{sentiment_probabilities}")

    aggregated_score = aggregate_sentiment_score(sentiment_probabilities)
    print(f"\nStep 2: Aggregated Sentiment Score: {aggregated_score:.4f}")

    predicted_difference = predict_price_change(aggregated_score, previous_price)
    print(f"\nStep 3: Final Predicted Price Difference: {predicted_difference:.4f}")

    return predicted_difference


# --- Example Execution ---
print("--- Running Full Pipeline Test ---")

sample_news = [
    "EIA reports a larger-than-expected build in natural gas storage, signaling oversupply.",
    "Geopolitical tensions in Eastern Europe threaten to disrupt key gas supply routes.",
    "A mild winter forecast across North America reduces expected heating demand."
]
previous_price = 2.75

print(f"Input News Articles: {len(sample_news)}")
print(f"Previous Day's Price: ${previous_price:.2f}\n")

final_prediction = natural_gas_pipeline(sample_news, previous_price)

print(f"\n--- Pipeline Complete ---")
print(f"Final Predicted Price Difference: {final_prediction:.4f}")
print(f"Predicted Tomorrow's Price: ${previous_price + final_prediction:.2f}")

## 4. Load News and Price Data

In [ ]:
import pandas as pd

# Load the news data from the JSON file (format: [{headline, date, summary}])
news_df = pd.read_json('gs://codeml/gas_headlines.json')

# Load the price data from the CSV file
prices_df = pd.read_csv('gs://codeml/DHHNGSP.csv')

# Display the first 5 rows of each DataFrame
print("News Data:")
display(news_df.head())
print(f"News columns: {news_df.columns.tolist()}")

print("\nPrice Data:")
display(prices_df.head())

## 5. Preprocess and Merge Data

In [ ]:
# 1. Convert date columns to datetime objects
news_df['date'] = pd.to_datetime(news_df['date'])
prices_df['observation_date'] = pd.to_datetime(prices_df['observation_date'])

# 2. Clean the prices_df
prices_df['DHHNGSP'] = pd.to_numeric(prices_df['DHHNGSP'], errors='coerce')
prices_df['DHHNGSP'] = prices_df['DHHNGSP'].ffill()

# 3. Create 'actual_tomorrow_price' column
prices_df['actual_tomorrow_price'] = prices_df['DHHNGSP'].shift(-1)

# 4. Combine headline and summary into full text for sentiment analysis
# Use both fields to get richer context
news_df['full_text'] = news_df.apply(
    lambda row: f"{row['headline']}. {row['summary']}" if pd.notna(row['summary']) and row['summary'] else row['headline'],
    axis=1
)

# 5. Group news by date (use combined headline + summary)
news_grouped_df = news_df.groupby('date')['full_text'].apply(list).reset_index()

# 6. Merge the dataframes
merged_df = pd.merge(news_grouped_df, prices_df, left_on='date', right_on='observation_date')

# 7. Rename the columns
merged_df.rename(columns={'observation_date': 'today_news_date', 'DHHNGSP': 'today_price'}, inplace=True)

# 8. Select and reorder the final columns
final_df = merged_df[['today_news_date', 'today_price', 'actual_tomorrow_price', 'full_text']]

# Display the first few rows
print("Merged and Preprocessed Data:")
display(final_df.head())
print(f"\nTotal rows: {len(final_df)}")
print(f"\nSample combined text for first date:")
print(f"First article: {final_df.iloc[0]['full_text'][0][:200]}..." if len(final_df.iloc[0]['full_text']) > 0 else "No articles")

## 6. Generate Predictions for All Data

In [ ]:
from tqdm import tqdm

# Initialize a list to store the results
predictions = []

# Process the entire DataFrame
for index, row in tqdm(final_df.iterrows(), total=len(final_df), desc="Processing news and predicting prices"):
    news_texts = row['full_text']
    today_price = row['today_price']

    # Ensure there are news texts to process
    if news_texts and isinstance(news_texts, list) and len(news_texts) > 0:
        sentiment_probabilities = get_sentiment(news_texts)
        aggregated_score = aggregate_sentiment_score(sentiment_probabilities)
    else:
        # If there are no news texts, assume a neutral sentiment
        aggregated_score = 0

    # Predict the price change and the next day's price
    predicted_difference = predict_price_change(aggregated_score, today_price)
    predicted_tomorrow_price = today_price + predicted_difference

    # Append the results
    predictions.append({
        'today_news_date': row['today_news_date'],
        'today_price': today_price,
        'predicted_tomorrow_price': predicted_tomorrow_price,
        'actual_tomorrow_price': row['actual_tomorrow_price'],
        'aggregated_score': aggregated_score
    })

# Convert the list of prediction dictionaries into a DataFrame
predictions_df = pd.DataFrame(predictions)

print("\nPrice Predictions:")
display(predictions_df.head(10))
print(f"\nTotal predictions: {len(predictions_df)}")

## 7. Save Predictions to CSV

In [ ]:
# Save the predictions to a CSV file
predictions_df.to_csv('price_predictions.csv', index=False)

print("Predictions saved to price_predictions.csv")
print(f"Total rows saved: {len(predictions_df)}")

## 8. Evaluate Model Performance

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Drop rows where 'actual_tomorrow_price' is NaN (last day in dataset)
predictions_eval_df = predictions_df.dropna(subset=['actual_tomorrow_price'])

# Extract the actual and predicted values
y_true = predictions_eval_df['actual_tomorrow_price']
y_pred = predictions_eval_df['predicted_tomorrow_price']

# Calculate MAE and RMSE
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("="*60)
print("MODEL PERFORMANCE EVALUATION")
print("="*60)
print(f"Total predictions evaluated: {len(predictions_eval_df)}")
print(f"\nMean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Performance Score: {100 / (1 + rmse):.2f}")
print("="*60)

## Summary

### Pipeline Overview
1. **Data Loading**: Natural gas news and historical price data were loaded from Google Cloud Storage
2. **Preprocessing**: Data was cleaned, dates were aligned, and news articles were grouped by date
3. **Text Combination**: Headlines and summaries were combined to provide richer context for sentiment analysis
4. **Sentiment Analysis**: FinBERT model (`yiyanghkust/finbert-tone`) analyzed sentiment of combined news text
5. **Price Prediction**: Sentiment scores were used to predict next day's price changes
6. **Evaluation**: Model performance was measured using MAE and RMSE metrics

### Key Findings
- The pipeline successfully processes all available data where news and price information overlap
- Both headlines AND summaries are used for sentiment analysis for better context
- Sentiment scores are calculated as the difference between positive and negative probabilities
- Price change formula: `predicted_difference = aggregated_score * abs(aggregated_score) * scaling_factor`
- Output CSV contains: date, today's price, predicted tomorrow's price, actual tomorrow's price, aggregated score

### Next Steps for Improvement
1. **Model Enhancement**: Use optimized magic multiplier from `get_magic_mult.py`
2. **Feature Engineering**: Add more features like trading volume, weather data, storage levels
3. **Time Series Models**: Experiment with ARIMA, LSTM, or Transformer-based forecasting
4. **Hyperparameter Tuning**: Optimize the sentiment-to-price scaling factor using historical data
5. **Cross-Validation**: Implement time-series cross-validation for more robust evaluation